In [3]:
!pip install pygame scikit-learn -q

import pygame
import random
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
#Game settings
WIDTH, HEIGHT = 800, 500

PADDLE_WIDTH = 15
PADDLE_HEIGHT = 90
PADDLE_SPEED = 6

BALL_SIZE = 15
BALL_SPEED = 5

FPS = 60

In [5]:
def rule_based_action(ball_y, paddle_y):
    paddle_center = paddle_y + PADDLE_HEIGHT / 2
    
    if ball_y < paddle_center - 10:
        return 0   # move up
    elif ball_y > paddle_center + 10:
        return 2   # move down
    else:
        return 1   # stay


def generate_training_data(n_samples=5000):
    X = []
    y = []
    
    for _ in range(n_samples):
        ball_x = random.randint(0, WIDTH)
        ball_y = random.randint(0, HEIGHT)
        ball_dx = random.choice([-BALL_SPEED, BALL_SPEED])
        ball_dy = random.choice([-BALL_SPEED, BALL_SPEED])
        paddle_y = random.randint(0, HEIGHT - PADDLE_HEIGHT)
        
        features = [
            ball_x / WIDTH,
            ball_y / HEIGHT,
            ball_dx / BALL_SPEED,
            ball_dy / BALL_SPEED,
            paddle_y / HEIGHT
        ]
        
        action = rule_based_action(ball_y, paddle_y)
        
        X.append(features)
        y.append(action)
        
    return np.array(X), np.array(y)


X_train, y_train = generate_training_data()

mlp_ai = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    max_iter=500,
    random_state=42
)

mlp_ai.fit(X_train, y_train)

print("MLP AI training completed.")

MLP AI training completed.


In [9]:
#Game Functions
def get_difficulty_settings(difficulty):
    if difficulty == "easy":
        return {
            "ai_speed": 3,
            "reaction_error": 70,
            "reaction_delay": 12
        }
    elif difficulty == "medium":
        return {
            "ai_speed": 5,
            "reaction_error": 35,
            "reaction_delay": 6
        }
    elif difficulty == "hard":
        return {
            "ai_speed": 6,
            "reaction_error": 10,
            "reaction_delay": 2
        }
    else:
        raise ValueError("Choose difficulty: easy, medium, or hard")


def run_pong(ai_type="rule", difficulty="medium", max_points=5, show_game=True):
    pygame.init()
    
    settings = get_difficulty_settings(difficulty)
    ai_speed = settings["ai_speed"]
    reaction_error = settings["reaction_error"]
    reaction_delay = settings["reaction_delay"]
    
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption(f"Pong - {ai_type.upper()} AI - {difficulty.upper()}")
    
    clock = pygame.time.Clock()
    font = pygame.font.SysFont("Arial", 36)
    
    WHITE = (255, 255, 255)
    BLACK = (0, 0, 0)
    
    player_x = 30
    player_y = HEIGHT // 2 - PADDLE_HEIGHT // 2
    
    ai_x = WIDTH - 45
    ai_y = HEIGHT // 2 - PADDLE_HEIGHT // 2
    
    ball_x = WIDTH // 2
    ball_y = HEIGHT // 2
    ball_dx = random.choice([-BALL_SPEED, BALL_SPEED])
    ball_dy = random.choice([-BALL_SPEED, BALL_SPEED])
    
    player_score = 0
    ai_score = 0
    
    rallies = []
    current_rally = 0
    ai_hits = 0
    ai_misses = 0
    
    frame_count = 0
    ai_target_y = ball_y
    
    running = True
    
    while running:
        clock.tick(FPS)
        frame_count += 1
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        
        keys = pygame.key.get_pressed()
        
        if keys[pygame.K_UP] and player_y > 0:
            player_y -= PADDLE_SPEED
        if keys[pygame.K_DOWN] and player_y < HEIGHT - PADDLE_HEIGHT:
            player_y += PADDLE_SPEED
        
        # AI only updates its target after some delay
        if frame_count % reaction_delay == 0:
            ai_target_y = ball_y + random.randint(-reaction_error, reaction_error)
        
        # AI decision
        if ai_type == "rule":
            action = rule_based_action(ai_target_y, ai_y)
            
        elif ai_type == "mlp":
            features = np.array([[
                ball_x / WIDTH,
                ai_target_y / HEIGHT,
                ball_dx / BALL_SPEED,
                ball_dy / BALL_SPEED,
                ai_y / HEIGHT
            ]])
            action = mlp_ai.predict(features)[0]
        
        if action == 0 and ai_y > 0:
            ai_y -= ai_speed
        elif action == 2 and ai_y < HEIGHT - PADDLE_HEIGHT:
            ai_y += ai_speed
        
        ai_y = max(0, min(ai_y, HEIGHT - PADDLE_HEIGHT))
        
        # Ball movement
        ball_x += ball_dx
        ball_y += ball_dy
        
        if ball_y <= 0 or ball_y >= HEIGHT - BALL_SIZE:
            ball_dy *= -1
        
        player_rect = pygame.Rect(player_x, player_y, PADDLE_WIDTH, PADDLE_HEIGHT)
        ai_rect = pygame.Rect(ai_x, ai_y, PADDLE_WIDTH, PADDLE_HEIGHT)
        ball_rect = pygame.Rect(ball_x, ball_y, BALL_SIZE, BALL_SIZE)
        
        if ball_rect.colliderect(player_rect):
            ball_dx *= -1
            current_rally += 1
        
        if ball_rect.colliderect(ai_rect):
            ball_dx *= -1
            current_rally += 1
            ai_hits += 1
        
        # Scoring
        if ball_x < 0:
            ai_score += 1
            rallies.append(current_rally)
            current_rally = 0
            
            ball_x = WIDTH // 2
            ball_y = HEIGHT // 2
            ball_dx = BALL_SPEED
            ball_dy = random.choice([-BALL_SPEED, BALL_SPEED])
        
        if ball_x > WIDTH:
            player_score += 1
            ai_misses += 1
            rallies.append(current_rally)
            current_rally = 0
            
            ball_x = WIDTH // 2
            ball_y = HEIGHT // 2
            ball_dx = -BALL_SPEED
            ball_dy = random.choice([-BALL_SPEED, BALL_SPEED])
        
        if show_game:
            screen.fill(BLACK)
            
            pygame.draw.rect(screen, WHITE, player_rect)
            pygame.draw.rect(screen, WHITE, ai_rect)
            pygame.draw.ellipse(screen, WHITE, ball_rect)
            pygame.draw.aaline(screen, WHITE, (WIDTH // 2, 0), (WIDTH // 2, HEIGHT))
            
            score_text = font.render(f"{player_score}   {ai_score}", True, WHITE)
            screen.blit(score_text, (WIDTH // 2 - 50, 20))
            
            label = font.render(f"{ai_type.upper()} AI - {difficulty.upper()}", True, WHITE)
            screen.blit(label, (WIDTH // 2 - 170, HEIGHT - 50))
            
            pygame.display.flip()
        
        if player_score >= max_points or ai_score >= max_points:
            running = False
    
    pygame.quit()
    
    total_ai_attempts = ai_hits + ai_misses
    hit_rate = ai_hits / total_ai_attempts if total_ai_attempts > 0 else 0
    average_rally = np.mean(rallies) if rallies else 0
    
    results = {
        "AI Type": ai_type,
        "Difficulty": difficulty,
        "Player Score": player_score,
        "AI Score": ai_score,
        "AI Hits": ai_hits,
        "AI Misses": ai_misses,
        "AI Hit Rate": round(hit_rate, 2),
        "Average Rally Length": round(average_rally, 2)
    }
    
    return results

In [10]:
easy_rule = run_pong(ai_type="rule", difficulty="easy", max_points=5)
easy_rule

{'AI Type': 'rule',
 'Difficulty': 'easy',
 'Player Score': 5,
 'AI Score': 1,
 'AI Hits': 0,
 'AI Misses': 5,
 'AI Hit Rate': 0.0,
 'Average Rally Length': np.float64(0.67)}

In [11]:
medium_mlp = run_pong(ai_type="mlp", difficulty="medium", max_points=5)
medium_mlp

{'AI Type': 'mlp',
 'Difficulty': 'medium',
 'Player Score': 5,
 'AI Score': 1,
 'AI Hits': 3,
 'AI Misses': 5,
 'AI Hit Rate': 0.38,
 'Average Rally Length': np.float64(4.33)}

In [12]:
hard_mlp = run_pong(ai_type="mlp", difficulty="hard", max_points=5)
hard_mlp

{'AI Type': 'mlp',
 'Difficulty': 'hard',
 'Player Score': 0,
 'AI Score': 5,
 'AI Hits': 36,
 'AI Misses': 0,
 'AI Hit Rate': 1.0,
 'Average Rally Length': np.float64(22.4)}

In [13]:
comparison = pd.DataFrame([easy_rule, medium_mlp, hard_mlp])
comparison

,AI Type,Difficulty,Player Score,AI Score,AI Hits,AI Misses,AI Hit Rate,Average Rally Length
0,rule,easy,5,1,0,5,0.00,0.67
1,mlp,medium,5,1,3,5,0.38,4.33
2,mlp,hard,0,5,36,0,1.00,22.40
